<a href="https://colab.research.google.com/github/nallamothu-varun/AI_Projects/blob/main/Project_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip3 install openai chromadb python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 108.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 97.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 121.9 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      S

In [2]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import chromadb

In [4]:
from google.colab import userdata
openai_api_key = userdata.get('OpenApi')
print(openai_api_key[:10])

sk-proj-gA


In [5]:
client = OpenAI(api_key = openai_api_key)
print("OpenAI client created successfully")

OpenAI client created successfully


Communicating to the Openai API

In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini", '''gpt-6-astra'''
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is RAG? Explain in 3 short bullet points"},
    ]
)
print(response.choices[0].message.content)

- **RAG (Retrieval-Augmented Generation)** helps AI answer questions using information retrieved from external sources.
- It finds relevant content—such as documents or database entries—and includes it as context when generating an answer.
- This can make answers more accurate, up to date, and grounded in sources, though mistakes are still possible.


In [12]:
response.model_dump_json()

'{"id":"chatcmpl-Edrkf2UlgkltRXdyY1gpJGTRsm731","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"- **RAG (Retrieval-Augmented Generation)** helps AI answer questions using information retrieved from external sources.\\n- It finds relevant content—such as documents or database entries—and includes it as context when generating an answer.\\n- This can make answers more accurate, up to date, and grounded in sources, though mistakes are still possible.","refusal":null,"role":"assistant","annotations":[],"audio":null,"function_call":null,"tool_calls":null}}],"created":1793170121,"model":"gpt-6-astra","object":"chat.completion","moderation":null,"service_tier":"default","system_fingerprint":null,"usage":{"completion_tokens":70,"prompt_tokens":28,"total_tokens":98,"completion_tokens_details":{"accepted_prediction_tokens":0,"audio_tokens":0,"reasoning_tokens":0,"rejected_prediction_tokens":0},"prompt_tokens_details":{"audio_tokens":0,"cache_write_tokens":0,"ca

Loading and chunking the data

In [14]:
with open("/content/company_hr_policy.txt", "r") as f:
    hr_document = f.read()
print(hr_document)

NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

SECTION 1: LEAVE POLICY

Annual Leave:
All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.

Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.

Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual leave cannot be taken for more than 3 consecutive days. Prior approval from the reporting man

In [15]:
print(f"Hr Document: {len(hr_document)}")
print(f"Number of words: {len(hr_document.split())}")

Hr Document: 7464
Number of words: 1052


Chunking strategy

In [17]:
def chunk_text(text, source_name):
  #split on double lines
  paragraph = text.strip().split("\n\n")
  chunks = []
  for para in paragraph:
    #split on single lines
    sentences = para.strip()
    if len(para) < 50:
      continue
    if para.startswith("=="):
      continue
    chunks.append({"text": para, "source": source_name})
  return chunks

hr_chunks = chunk_text(hr_document, "HR Policy")
print(f"Number of chunks: {len(hr_chunks)}")

Number of chunks: 25


Storing Chunks with ChromaDB

In [20]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="hr_policy")

In [18]:
documents=[]
ids=[]
metadata=[]
for i, chunk in enumerate(hr_chunks):
  documents.append(chunk["text"])
  ids.append(f"chunk_{i}")
  metadata.append({"source": chunk["source"]})

In [21]:
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 81.2MiB/s]


Building Retrival Pipeline

In [24]:
def retrieve(question, n_results=3):
  results = collection.query(
      query_texts=question,
      n_results=n_results,
  )
  return results['documents'][0], results['metadatas'][0]
#

In [27]:
chunks, sources = retrieve("What is the work from home policy?", 3)
for i in range(len(chunks)):
  print(f"Chunks{i+1}")
  print(f"source: {sources[i]}")
  print(f"Text: {chunks[i]}")
  print()

Chunks1
source: {'source': 'HR Policy'}
Text: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.

Chunks2
source: {'source': 'HR Policy'}
Text: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

Chunks3
source: {'source': 'HR Policy'}
Text: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days.



In [28]:
def ask_rag(question, n_results=3, verbose=True):
  chunks, sources = retrieve(question, n_results)
  if verbose:
    print(f"Sources: {sources}")
  context = "\n\n".join(chunks)
  messages = [
      {
          "role": "system",
          "content": (
              "you are a helpful assistant that answers questions based on the provided context"
              "If the context does not contain information to answer the question, say 'I don't know' "
          )
      },
      {
        "role": "user",
        "content": f"Context: {context}\n\n---\n\nQuestion:{question}"
      }
  ]
  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=messages,
      temperature = 0.2
  )

  answer = response.choices[0].message.content
  if verbose:
    print(f"Answer: {answer}")
    print(f"{'='*60}")
  return answer
print("Rag pipeline created successfully")

Rag pipeline created successfully


In [29]:
ask_rag("What is the work from home policy?")

Sources: [{'source': 'HR Policy'}, {'source': 'HR Policy'}, {'source': 'HR Policy'}]
Answer: The work from home (WFH) policy at NovaTech Solutions allows all employees who have completed their probation period of 6 months to work from home. Employees may work from home up to 2 days per week, with preferred days being Wednesday and Friday, although teams can adjust these based on project needs. Employees must be available during core working hours from 10:00 AM to 6:00 PM IST on WFH days. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.


'The work from home (WFH) policy at NovaTech Solutions allows all employees who have completed their probation period of 6 months to work from home. Employees may work from home up to 2 days per week, with preferred days being Wednesday and Friday, although teams can adjust these based on project needs. Employees must be available during core working hours from 10:00 AM to 6:00 PM IST on WFH days. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.'